In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
os.environ["LANGSMITH_API_KEY"] = os.getenv("LANGSMITH_API_KEY")
os.environ["LANGSMITH_TRACKING"] = "true"

# Avoid printing secrets from environment variables in notebooks.

#Create a data point for evaluation
from langsmith import Client
client = Client()

# Define the dataset name and create a new dataset using the LangSmith client
dataset_name = "LangSmith Evaluation Dataset4"
dataset = client.create_dataset(dataset_name)

examples=[
  # --- Engineering meetings (file 1) ---
  {
    "input": "Who was responsible for reviewing the Jetpack Compose migration?",
    "output": "Michael Brown, the Android Tech Lead, was assigned to review the Compose migration progress in the Android Application Performance Review meeting."
  },
  {
    "input": "What decisions were made in the Database Optimization Discussion meeting?",
    "output": "The team decided to add indexes to frequently searched columns, archive old transactional data, and enable database query monitoring."
  },
  {
    "input": "Which meeting discussed cloud infrastructure optimization and what were the decisions?",
    "output": "The Cloud Infrastructure Review meeting (July 6, 2026) decided to remove unused environments, enable automatic scaling, and review monthly cloud expenses."
  },
  {
    "input": "What improvements were planned for the Android application?",
    "output": "The team planned to migrate remaining screens to Jetpack Compose, improve image caching implementation, and reduce unnecessary network requests."
  },
  {
    "input": "Who was assigned to optimize SQL queries?",
    "output": "Sarah Lee was assigned to optimize the SQL queries in the Database Optimization Discussion meeting."
  },
  {
    "input": "What decisions were made in the Security Review Meeting?",
    "output": "The team decided to enable additional API rate limiting and improve authentication logging."
  },
  {
    "input": "What was decided during the API Integration Planning meeting?",
    "output": "The team decided that API integration will happen in two phases, sandbox testing must complete before production deployment, and additional monitoring will be added."
  },
  {
    "input": "What was decided in the Mobile Release Readiness meeting?",
    "output": "The release candidate will be submitted after final QA, and crash monitoring will remain enabled after launch."
  },
  {
    "input": "What action items were assigned to David Wilson?",
    "output": "David Wilson was assigned to prepare the migration plan in the Backend Architecture Review meeting."
  }
]

client.create_examples(
    inputs=[
        {"question": ex["input"]}
        for ex in examples
    ],
    outputs=[
        {"answer": ex["output"]}
        for ex in examples
    ],
    dataset_id=dataset.id,
)


In [23]:
from dotenv import load_dotenv
from pathlib import Path
import json
from langchain_openai import ChatOpenAI


BASE_DIR = Path.cwd()
load_dotenv(BASE_DIR / ".env")


model = os.getenv("LLM_MODEL", "gpt-4o-mini")
print(f"Using model: {model}", flush=True)

judge_llm = ChatOpenAI(
    model=model,
    api_key=os.getenv("OMNIROUTER_API_KEY"),
    base_url=os.getenv("OMNIROUTER_BASE_URL"),
    streaming=True,
)

Using model: auto/best-coding


In [24]:
import openai
from langsmith import wrappers

openrouter_client = openai.OpenAI(
    api_key=os.getenv("OMNIROUTER_API_KEY"),
    base_url=os.getenv("OMNIROUTER_BASE_URL")
)
 
openai_client=wrappers.wrap_openai(openrouter_client)



In [25]:
eval_prompt_system = """
You are evaluating whether an AI answer correctly answers a user's question.

Compare the user's question, the reference answer, and the AI answer.

Guidelines:
- Judge correctness with respect to the user's question.
- The reference answer is an example of a correct answer, not the only acceptable one.
- Accept paraphrases and alternative correct answers.
- Ignore differences in wording, formatting, or level of detail.
- Additional correct information is acceptable.
- Ignore omitted secondary details unless they are necessary to answer the question.
- Return INCORRECT only if the AI:
  - gives incorrect facts,
  - contradicts the correct answer,
  - fails to answer the question, or
  - omits essential information needed to answer correctly.

Return exactly one word:

CORRECT

or

INCORRECT
"""

In [26]:
def correctness(
    inputs: dict,
    outputs: dict,
    reference_outputs: dict
) -> bool:

    prompt = f"""
        Question:
        {inputs['question']}

        Reference Answer:
        {reference_outputs['answer']}

        AI Answer:
        {outputs['response']}
"""

    result = judge_llm.invoke(
        [
            ("system", eval_prompt_system),
            ("human", prompt)
        ]
    )

    return result.content.strip().upper() == "CORRECT"

In [27]:
faithfulness_system_prompt = """
You are evaluating the faithfulness of an AI-generated response.

Your task is to determine whether every factual claim in the AI response is supported by the retrieved source documents.

Rules:
- The retrieved source documents are the only source of truth.
- Ignore the reference answer when judging faithfulness.
- Evaluate only factual claims.
- Ignore writing quality, style, formatting, and organization.
- Paraphrases are acceptable.
- A claim is supported if it is explicitly stated or is a direct logical consequence of the retrieved documents.
- Do not use external knowledge.
- Do not penalize omitted information.
- If any factual claim is unsupported or contradicts the retrieved documents, return UNFAITHFUL.
- Otherwise return FAITHFUL.

Output exactly one word:
FAITHFUL
or
UNFAITHFUL
"""

faithfulness_user_prompt = """
Retrieved Source Documents:
{source_documents}

Reference Answer (context only):
{reference_answer}

AI Response:
{generated_answer}

Determine whether every factual claim in the AI response is supported by the retrieved source documents.

Return exactly one word:

FAITHFUL

or

UNFAITHFUL
"""

In [28]:
def faithfulness(
    inputs: dict,
    outputs: dict,
    reference_outputs: dict
) -> bool:

    question = inputs["question"]

    # Retrieve the actual source documents from ChromaDB for this question
    source_documents = query_documents.invoke({"question": question})

    prompt = faithfulness_user_prompt.format(
        source_documents=source_documents,
        reference_answer=reference_outputs["answer"],
        generated_answer=outputs["response"]
    )

    result = judge_llm.invoke(
        [
            ("system", faithfulness_system_prompt),
            ("human", prompt)
        ]
    )

    verdict = result.content.strip().upper()

    print(f"Faithfulness Verdict: {verdict}", flush=True)

    return verdict == "FAITHFUL"

In [37]:
from tools import query_documents



def my_app(question: str) -> str:

    app_llm = ChatOpenAI(
        model=model,
        api_key=os.getenv("OMNIROUTER_API_KEY"),
        base_url=os.getenv("OMNIROUTER_BASE_URL"),
        streaming=True,
    )

    # Retrieve relevant meeting notes from Chroma
    context = query_documents.invoke({
        "question": question
    })

    prompt = f"""
    You are answering questions about meeting notes.

    Rules:
    - Use ONLY the meeting notes provided below.
    - If the answer exists in the notes, answer directly.
    - Retrieved chunks may belong to the same meeting.
    - Combine related chunks before answering.
    - Do not assume information is missing just because the meeting title and details are in different chunks.
    - Do not say "I couldn't find that information" when relevant information exists.
    - Only say "I couldn't find that information" if the notes truly do not contain the answer.

Meeting Notes:
{context}

Question:
{question}

"""

    try:
        result = app_llm.invoke(
            [
                ("system", "You answer questions from meeting notes."),
                ("human", prompt)
            ]
        )
    except ValueError:
        # Fallback: retry without streaming if the stream fails
        app_llm_fallback = ChatOpenAI(
            model=model,
            api_key=os.getenv("OMNIROUTER_API_KEY"),
            base_url=os.getenv("OMNIROUTER_BASE_URL"),
            streaming=False,
        )
        result = app_llm_fallback.invoke(
            [
                ("system", "You answer questions from meeting notes."),
                ("human", prompt)
            ]
        )

    return result.content.strip()

In [38]:
### Call my_app for every datapoints
def ls_target(inputs: str) -> dict:
    return {"response": my_app(inputs["question"])}

In [40]:
## Run our evaluation
experiment_results=client.evaluate(
    ls_target, ## Your AI system
    data=dataset_name,
    evaluators=[correctness,faithfulness],
    experiment_prefix="open-router-meeting-notes-chatbot"
)

View the evaluation results for experiment: 'open-router-meeting-notes-chatbot-d8f87bf3' at:
https://smith.langchain.com/o/04eb0163-f56b-478e-aed0-73810e4298b3/datasets/44a6d0fd-7f86-44aa-b5bb-30c48e01324a/compare?selectedSessions=2848f7bd-814e-4072-9373-ce79aa836b3a




0it [00:00, ?it/s]

Faithfulness Verdict: FAITHFUL


1it [00:35, 35.20s/it]

Faithfulness Verdict: FAITHFUL


...

7it [03:41, 32.84s/it]

Faithfulness Verdict: FAITHFUL


8it [04:19, 34.53s/it]

Faithfulness Verdict: FAITHFUL


9it [04:48, 32.01s/it]
